# MimicGuard v2 / 4: R3 harder attacker with raw-IQ oracle, R6 raw conv-AE variants (GPU)

Simulation-only. R3 repeats the headroom experiment at alpha = 0.75 (a better-mimicking attacker, so fewer cells saturate) and reports Pd within short horizons. R6 trains five raw conv-AE variants and reports all of them.

**Kaggle settings:** Accelerator = **GPU (T4 or P100)**; Internet = **Off** (nothing is downloaded; no token or secret is needed);
no dataset needs to be added. Run with *Save Version -> Save & Run All (Commit)* for the long runs so the output is kept.
Set `MODE` in the config cell: `"smoke"` = a couple of minutes, to check that everything works; `"medium"` and `"full"` = the real runs.
Runs are resumable: if the session stops, run it again and finished units are skipped (the output folder `results_r3_r6` must stay the same).

Estimated run time (not measured on Kaggle): R3 about 30-40 min and R6 about 20-30 min on a T4 (estimate from the notebook 3 wall time of 1894 s for 27 units). MODE `smoke` first (about 2-5 min).

In [ ]:
# ---- 1. Setup: unpack the embedded code, check the environment ----
import base64, io, os, sys, zipfile, subprocess, platform, warnings
warnings.filterwarnings("ignore")
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("mimicguard_run")
os.makedirs(WORK, exist_ok=True)
REPO = os.path.join(WORK, "mimicguard")
_B64 = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_B64))) as z:
    z.extractall(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
os.chdir(REPO)
try:
    import yaml
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)
import numpy, scipy, sklearn
print("python", platform.python_version(), "| numpy", numpy.__version__, "| sklearn", sklearn.__version__, "| CPUs", os.cpu_count())
try:
    import torch
    print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
except Exception as e:
    print("torch missing:", e)

In [ ]:
# ---- 2. Unit tests ----
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"], cwd=REPO, capture_output=True, text=True)
print(r.stdout[-1800:], r.stderr[-800:])
assert r.returncode == 0, "unit tests failed: stop and send the output above"

In [ ]:
# ---- 3. Config ----
from mimicguard.config import load_config
from mimicguard.v2 import experiments as ex, followups as fu, rawiq
import pandas as pd, numpy as np
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 300)
BASE = load_config(os.path.join(REPO, "experiments/configs/base.yaml"))
MODE = "full"                    # "smoke" | "full"
N_JOBS = os.cpu_count() or 1
OUT = os.path.join(WORK, "results_r3_r6"); os.makedirs(OUT, exist_ok=True)
TINY = dict(n_train=12, n_val=8, n_cal=20, n_chk=20, n_test=10, n_otrain=12, H=40, n_test_win=80, onset=40)

import torch
print("device:", "cuda" if torch.cuda.is_available() else "CPU (slow: turn on the GPU accelerator)")
BIG = dict(n_train=300, n_val=40, n_cal=500, n_chk=500, n_test=100, n_otrain=300, H=100, n_test_win=200, onset=100)
if MODE == "smoke":
    R3_CELLS, R3_SEEDS, R6_CELLS, R6_SEEDS, SIZES, AEK, ORK = [("rayleigh", 20)], [0], [("rayleigh", 20)], [0], TINY, dict(epochs=3), dict(epochs=3)
    VARIANTS = {"default": {"epochs": 3}, "latent4": {"latent": 4, "epochs": 3}}
else:
    R3_CELLS = [(f, s) for f in ("rayleigh", "rician", "none") for s in (15, 20, 25)]; R3_SEEDS = [0, 1, 2]
    R6_CELLS = [(f, 20) for f in ("rayleigh", "rician", "none")]; R6_SEEDS = [0, 1, 2]
    SIZES, AEK, ORK, VARIANTS = BIG, {}, {}, None
print(MODE, "R3", R3_CELLS, R3_SEEDS, "R6", R6_CELLS, R6_SEEDS)

In [ ]:
# ---- 4. R3: raw-IQ and six-feature detectors and oracles at alpha = 0.75 (harder attacker, fewer saturated cells) ----
r3 = rawiq.run_e2(BASE, R3_CELLS, R3_SEEDS, os.path.join(OUT, "r3"), SIZES, n_jobs=N_JOBS, alpha=0.75, ae_kwargs=AEK, oracle_kwargs=ORK, save_arrays=True)
r3.to_csv(os.path.join(OUT, "r3_rows.csv"), index=False)
print(len(r3), "rows")

In [ ]:
# ---- 5. R3 results: Pd at p = 0.05 and 0.01, Pd within 20 windows, CADD, window AUC ----
import glob
order = ["pca_L4", "lof", "ocsvm", "ae", "oracle_feat", "raw_conv_ae", "raw_conv_oracle"]; order += [o + "_cusum" for o in order]
for p in (0.05, 0.01):
    d = r3[r3.p == p]
    for m in ("pd", "cadd", "auc", "fa_chk"):
        print(f"=== {m} (p = {p}) ==="); print(d.groupby(["detector", "fading", "snr_db"])[m].mean().unstack(["fading", "snr_db"]).reindex(order).round(3))
# Pd within a short horizon, from the stored per-stream first-alarm delays (NaN = no alarm)
rows = []
for f in glob.glob(os.path.join(OUT, "r3", "e2_s*_*.npz")):
    stem = os.path.basename(f)[:-4]; _, s, fad, snr = stem.split("_"); z = np.load(f)
    for k in z.files:
        if k.endswith("|delay"):
            name, lab, p, _ = k.split("|"); dl = z[k]
            for H in (10, 20, 50, 100):
                rows.append(dict(seed=s, fading=fad, snr_db=float(snr), detector=name, p=float(p), horizon=H, pd_h=float(np.nanmean(np.where(np.isnan(dl), np.inf, dl) < H))))
sh = pd.DataFrame(rows); sh.to_csv(os.path.join(OUT, "r3_short_horizon.csv"), index=False)
for p in (0.05, 0.01):
    print(f"=== Pd within 20 windows (p = {p}) ==="); print(sh[(sh.p == p) & (sh.horizon == 20)].groupby(["detector", "fading", "snr_db"]).pd_h.mean().unstack(["fading", "snr_db"]).reindex(order).round(3))

In [ ]:
# ---- 6. R6: raw conv-AE variants (all reported; no variant is chosen with attack data) ----
r6 = fu.run_r6(BASE, R6_CELLS, R6_SEEDS, os.path.join(OUT, "r6"), SIZES, variants=VARIANTS, n_jobs=N_JOBS, alpha=0.5)
r6.to_csv(os.path.join(OUT, "r6_rows.csv"), index=False)
for m in ("pd", "auc", "fa_chk"):
    print(f"=== {m} (p = 0.05) ==="); print(r6[r6.p == 0.05].groupby(["detector", "fading"])[m].mean().unstack("fading").round(3))

In [ ]:
# ---- last cell: package outputs ----
import shutil
zp = shutil.make_archive(os.path.join(WORK, "mimicguard_v2_r3_r6_results"), "zip", OUT)
print("results archive:", zp, f"({os.path.getsize(zp)/1e6:.1f} MB)")
print("Download it from the Output tab (after Save Version -> Save & Run All), then send it back.")